In [1]:
import numpy as np
import pandas as pd

In [2]:
fold0 = pd.read_csv(f"/scratch1/smaruj/suppressing_CTCFs/results_repeated/fold0_with_positions_steps_results.tsv", sep="\t")

fold0["fold"] = [0 for i in range(len(fold0))]

In [3]:
fold1 = pd.read_csv(f"/scratch1/smaruj/suppressing_CTCFs/results_repeated/fold1_with_positions_steps_results.tsv", sep="\t")

fold1["fold"] = [1 for i in range(len(fold1))]

In [4]:
fold2 = pd.read_csv(f"/scratch1/smaruj/suppressing_CTCFs/results_repeated/fold2_with_positions_steps_results.tsv", sep="\t")

fold2["fold"] = [2 for i in range(len(fold2))]

In [5]:
fold3 = pd.read_csv(f"/scratch1/smaruj/suppressing_CTCFs/results_repeated/fold3_with_positions_steps_results.tsv", sep="\t")

fold3["fold"] = [3 for i in range(len(fold3))]

In [6]:
fold4 = pd.read_csv(f"/scratch1/smaruj/suppressing_CTCFs/results_repeated/fold4_with_positions_steps_results.tsv", sep="\t")

fold4["fold"] = [4 for i in range(len(fold4))]

In [7]:
fold5 = pd.read_csv(f"/scratch1/smaruj/suppressing_CTCFs/results_repeated/fold5_with_positions_steps_results.tsv", sep="\t")

fold5["fold"] = [5 for i in range(len(fold5))]

In [8]:
fold6 = pd.read_csv(f"/scratch1/smaruj/suppressing_CTCFs/results_repeated/fold6_with_positions_steps_results.tsv", sep="\t")

fold6["fold"] = [6 for i in range(len(fold6))]

In [9]:
fold7 = pd.read_csv(f"/scratch1/smaruj/suppressing_CTCFs/results_repeated/fold7_with_positions_steps_results.tsv", sep="\t")

fold7["fold"] = [7 for i in range(len(fold7))]

In [10]:
df = pd.concat([fold0, fold1, fold2,
                fold3, fold4, fold5,
                fold6, fold7], ignore_index=True)

In [11]:
df["URQ_diff"] = df["URQ_result"] - df["URQ_init"]

In [12]:
# selecting only sequences with a measurable contact enrichment
df = df[df['URQ_diff'] > 0.0]

In [ ]:
df.columns

In [13]:
import ast

# Convert stringified sets into real Python sets of tuples
df["orig_CTCFs_coord"] = df["orig_CTCFs_coord"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df["new_CTCFs_coord"] = df["new_CTCFs_coord"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

In [14]:
rows = []

for _, row in df.iterrows():
    chrom = row["chrom"]
    fold = row["fold"]
    centered_start = row["centered_start"]
    centered_end = row["centered_end"]
    
    coords = row["orig_CTCFs_coord"]
    # coords = row["new_CTCFs_coord"]
    
    if not coords:  # skip empty sets
        continue
    
    for coord in coords:
        if isinstance(coord, tuple) and len(coord) == 3:
            start, end, orientation = coord
            rows.append({
                "chrom": chrom,
                "fold": fold,
                "centered_start": centered_start,
                "centered_end": centered_end,
                "ctcf_start": start,
                "ctcf_end": end,
                "orientation": orientation
            })

ctcf_df = pd.DataFrame(rows)

In [15]:
ctcf_df

,chrom,fold,centered_start,centered_end,ctcf_start,ctcf_end,orientation
0,chr1,0,37799936,39110656,1601,1620,+
1,chr1,0,37799936,39110656,6,25,-
2,chr1,0,37799936,39110656,586,605,+
3,chr11,0,65677312,66988032,1400,1419,+
4,chr11,0,65677312,66988032,1487,1506,+
...,...,...,...,...,...,...,...
899,chrX,7,166264832,167575552,1781,1800,+
900,chrX,7,166264832,167575552,108,127,-
901,chrX,7,99522560,100833280,1607,1626,-
902,chrX,7,99522560,100833280,1463,1482,-


In [16]:
ctcf_df[ctcf_df["ctcf_end"] > 2048]

,chrom,fold,centered_start,centered_end,ctcf_start,ctcf_end,orientation
49,chr4,0,127655936,128966656,2075,2094,+
75,chr5,0,97955840,99266560,2031,2050,-
156,chr13,1,39806976,41117696,2038,2057,+
157,chr13,1,39806976,41117696,2057,2076,+
357,chr9,2,99528704,100839424,2053,2072,-
406,chr13,3,100675584,101986304,2033,2052,-
820,chr11,7,92659712,93970432,2031,2050,-
864,chr19,7,51900416,53211136,2080,2099,+


In [ ]:
# ctcf_df.to_csv("/scratch1/smaruj/suppressing_CTCFs/results_repeated/preexisting_ctcf_df.tsv", sep="\t", index=False)
ctcf_df.to_csv("/scratch1/smaruj/suppressing_CTCFs/results_repeated/new_ctcf_df.tsv", sep="\t", index=False)

# pred

In [ ]:
import torch
import numpy as np
import pandas as pd
from Bio import SeqIO

In [ ]:
import sys
import os
sys.path.append(os.path.abspath("/home1/smaruj/pytorch_akita/"))

from model_v2_compatible import SeqNN

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
model = SeqNN()
model.load_state_dict(torch.load("/home1/smaruj/pytorch_akita/model_0_v2_finetuned_correctly.pt", map_location=device))
model.eval()

In [ ]:
def one_hot_encode(seq, alphabet="ACGT"):
    """One-hot encode a DNA sequence into (len(seq), 4) numpy array."""
    mapping = {base: i for i, base in enumerate(alphabet)}
    arr = np.zeros((len(seq), len(alphabet)), dtype=np.float32)
    for i, base in enumerate(seq.upper()):
        if base in mapping:
            arr[i, mapping[base]] = 1.0
    return arr

In [ ]:
# Load ctcf table
df = pd.read_csv(
    "/scratch1/smaruj/suppressing_CTCFs/results/preexisting_ctcf_df.tsv", 
    sep="\t"
)

In [ ]:
# Load background fasta (first sequence only)
background_fasta = "/scratch1/smaruj/background_generation/background_sequences_scd30_totvar1300.fasta"
background_record = next(SeqIO.parse(background_fasta, "fasta"))
background_seq = str(background_record.seq)
print(f"Background length: {len(background_seq)}")

In [ ]:
# One-hot encode background
background_1hot = one_hot_encode(background_seq)
background_tensor = torch.tensor(background_1hot.T).unsqueeze(0).to(device)  # shape (1, 4, L)

In [ ]:
# Baseline prediction
with torch.no_grad():
    baseline_pred = model(background_tensor).cpu()

In [ ]:
df

In [ ]:
df[df["ctcf_start"] < 0]

In [ ]:
df[df["ctcf_end"] > 2048]

In [ ]:
def reverse_complement(ohe_seq):
    """
    Reverse complement of one-hot encoded sequence.
    Input shape: (B, 4, L) or (4, L).
    Returns same shape.
    """
    if ohe_seq.ndim == 2:  # (4, L) → add batch dim
        ohe_seq = ohe_seq.unsqueeze(0)

    # Reverse along sequence axis (last dimension)
    rc = torch.flip(ohe_seq, dims=[-1])

    # Complement: swap A<->T, C<->G along channel axis (dim=1)
    rc = rc[:, [3, 2, 1, 0], :]

    if rc.shape[0] == 1:  # if we added batch dim, squeeze it back
        rc = rc.squeeze(0)

    return rc

In [ ]:

import os
from tqdm import tqdm

extra_flank = 60
bin_size = 2048
center_bin = 320  # central bin index

slice_start = bin_size * center_bin
slice_end = bin_size * (center_bin+1)

collected_ctcfs = []

for fold in range(1):
    fold_df = df[df["fold"] == fold]
    
    for _, row in tqdm(fold_df.iterrows(), total=len(fold_df), desc=f"Fold {fold}"):
        chrom = row["chrom"]
        cstart = row["centered_start"]
        cend = row["centered_end"]
        ctcf_start = row["ctcf_start"]
        ctcf_end = row["ctcf_end"]
        orientation = row["orientation"]

        # Path to OHE file
        path = f"/scratch1/smaruj/suppressing_CTCFs/results/fold{fold}/{chrom}_{cstart}_{cend}_slice.pt"
        if not os.path.exists(path):
            print(f"⚠️ Missing: {path}")
            continue

        # Load OHE tensor (4, L)
        slice_ohe = torch.load(path).to(device)
        
        if ctcf_start >= 0 and ctcf_end <= 2048:
            # fully inside slice → take directly
            ctcf_seq = slice_ohe[:, :, ctcf_start:ctcf_end].squeeze(0)
            
        else:
            parts = []

            full_path = f"/scratch1/smaruj/suppressing_CTCFs/ohe_X/fold{fold}/{chrom}_{cstart}_{cend}_X.pt"
            if not os.path.exists(full_path):
                print(f"⚠️ Missing: {full_path}")
                continue

            # Load OHE tensor (4, L)
            full_ohe = torch.load(full_path).to(device)
            
            # left overhang
            if ctcf_start < 0:
                # how much overhang
                left_len = -ctcf_start
                # take from full sequence before slice_start
                parts.append(full_ohe[:, :, slice_start+ctcf_start : slice_start])

                # take the inside part from slice (from 0 to ctcf_end)
                parts.append(slice_ohe[:, :, 0:ctcf_end])

            # right overhang
            elif ctcf_end > 2048:
                right_len = ctcf_end - 2048
                # take the inside part from slice (from ctcf_start to 2048)
                parts.append(slice_ohe[:, :, ctcf_start:2048])
                # take from full sequence after slice_end
                parts.append(full_ohe[:, :, slice_end : slice_end+right_len])

            ctcf_seq = torch.cat(parts, dim=2)

        # Reverse complement if needed
        if orientation == "-":
            ctcf_seq = reverse_complement(ctcf_seq)
        
        collected_ctcfs.append(ctcf_seq)

In [ ]:
batch_size = 16
scd_scores = []

batch_tensors = []
batch_indices = []

for idx, ctcf_ohe in enumerate(collected_ctcfs):  
    # Ensure shape is (4, L_ctcf) (remove batch dim if present)
    if ctcf_ohe.ndim == 3:  
        ctcf_ohe = ctcf_ohe[0]

    # Clone background (shape (4, L_bg))
    inserted_tensor = background_tensor.clone()

    # Insert position (middle of background)
    insert_pos = inserted_tensor.shape[2] // 2
    frag_len = ctcf_ohe.shape[1]

    start = insert_pos - frag_len // 2
    end = start + frag_len
    
    # Insert CTCF sequence
    inserted_tensor[:, :, start:end] = ctcf_ohe.unsqueeze(0)

    # Collect for batching
    batch_tensors.append(inserted_tensor.to(device))
    batch_indices.append(idx)

    # Process batch
    if len(batch_tensors) == batch_size or idx == len(collected_ctcfs) - 1:
        batch_input = torch.cat(batch_tensors, dim=0)  # (B, 4, L)
        with torch.no_grad():
            batch_pred = model(batch_input).cpu()

        # Compute SCD
        for i, pred in enumerate(batch_pred):
            diff = pred - baseline_pred.squeeze(0)
            scd = torch.sqrt(torch.sum(diff ** 2)).item()
            scd_scores.append((batch_indices[i], scd))

        # Reset
        batch_tensors, batch_indices = [], []


In [ ]:
scd_scores